In [9]:
import os
import json
import time
import joblib
import warnings
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score

import xgboost as xgb
from flask import Flask, request, jsonify

In [10]:
warnings.filterwarnings("ignore")

# ================= CONFIG =================
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

FEATURES = [
    "V1","V2","V3","V4","V5","V6","V7","V8","V9","V10",
    "V11","V12","V13","V14","V15","V16","V17","V18","V19","V20",
    "V21","V22","V23","V24","V25","V26","V27","V28",
    "log_amount",
    "time_scaled"
]

In [11]:
def load_transactions(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    df = df.rename(columns={
        "Amount": "amount",
        "Class": "is_fraud",
        "Time": "time"
    })

    return df


In [12]:
def basic_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["log_amount"] = np.log1p(df["amount"])
    df["time_scaled"] = df["time"] / df["time"].max()

    return df

In [13]:
def train_models(data_path: str):
    print("Loading data...")
    df = load_transactions(data_path)
    df = basic_feature_engineering(df)

    X = df[FEATURES]
    y = df["is_fraud"].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25,
        stratify=y,
        random_state=RANDOM_STATE
    )

    # ---------- Scaling ----------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ---------- Isolation Forest ----------
    iso_model = IsolationForest(
        n_estimators=200,
        contamination=0.002,
        random_state=RANDOM_STATE
    )
    iso_model.fit(X_train_scaled)

    joblib.dump(
        {"scaler": scaler, "model": iso_model},
        MODEL_DIR / "isolation_forest.joblib"
    )
    print("Saved Isolation Forest")

    # ---------- XGBoost Supervised ----------
    clf = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )

    clf.fit(X_train_scaled, y_train)

    joblib.dump(
        {"scaler": scaler, "model": clf},
        MODEL_DIR / "xgb_supervised.joblib"
    )
    print("Saved XGBoost model")

    # ---------- Evaluation ----------
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]

    print("\nClassification Report")
    print(classification_report(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_prob))

    # ---------- Metadata ----------
    meta = {
        "threshold": 0.7,
        "contamination": 0.002
    }

    with open(MODEL_DIR / "meta.json", "w") as f:
        json.dump(meta, f)

    print("Training completed successfully")

In [14]:

# ================= RULE ENGINE =================
def apply_rule_engine(txn: Dict) -> Dict:
    rules = []

    if txn.get("amount", 0) > 2000:
        rules.append("HIGH_AMOUNT")

    return {
        "rules_triggered": rules,
        "rule_score": min(len(rules) * 0.3, 1.0)
    }


In [15]:
# ================= SCORING =================
def score_transaction(txn: Dict) -> Dict:
    iso_obj = joblib.load(MODEL_DIR / "isolation_forest.joblib")
    clf_obj = joblib.load(MODEL_DIR / "xgb_supervised.joblib")

    iso_scaler = iso_obj["scaler"]
    iso_model = iso_obj["model"]

    scaler = clf_obj["scaler"]
    clf = clf_obj["model"]

    with open(MODEL_DIR / "meta.json") as f:
        meta = json.load(f)

    df = pd.DataFrame([txn])
    df = basic_feature_engineering(df)

    X = df[FEATURES]
    X_scaled = scaler.transform(X)

    iso_score = float(iso_model.decision_function(X_scaled)[0])
    clf_prob = float(clf.predict_proba(X_scaled)[0][1])

    rules = apply_rule_engine(txn)

    overall_risk = min(
        1.0,
        0.6 * clf_prob +
        0.2 * (iso_score < -0.1) +
        0.2 * rules["rule_score"]
    )

    return {
        "iso_score": iso_score,
        "fraud_probability": clf_prob,
        "rules": rules,
        "overall_risk": overall_risk
    }

In [ ]:
def send_alert(txn_id, risk):
    print(f"[ALERT] Transaction {txn_id} flagged | Risk={risk:.2f}")


# ================= FLASK API =================
# ================= ALERT FUNCTION =================

def send_alert(txn_id, amount, risk):

    if risk >= 0.7:

        return {
            "decision": "REJECTED",
            "message": f"Transaction {txn_id} rejected! Amount ₹{amount}"
        }

    else:

        return {
            "decision": "APPROVED",
            "message": f"Transaction {txn_id} approved! Amount ₹{amount}"
        }


# ================= FLASK APP =================

app = Flask(__name__)


# ---------- HOME PAGE ----------

@app.route("/")
def home():

    return """
    <html>
    <head>
        <title>Fraud Detection System</title>
    </head>

    <body style="font-family: Arial; padding: 40px;">

        <h1>AI Fraud Detection System</h1>

        <form action="/predict" method="post">

            <label>Transaction ID:</label><br>
            <input type="text" name="transaction_id"><br><br>

            <label>Amount:</label><br>
            <input type="number" name="amount"><br><br>

            <button type="submit">
                Check Transaction
            </button>

        </form>

    </body>
    </html>
    """


# ---------- BROWSER PREDICTION ----------

@app.route("/predict", methods=["POST"])
def predict():

    transaction_id = request.form["transaction_id"]
    amount = float(request.form["amount"])

    # Dummy values for V1-V28
    txn = {
        "transaction_id": transaction_id,
        "amount": amount,
        "time": 50000
    }

    for i in range(1, 29):
        txn[f"V{i}"] = 0.0

    # Score transaction
    res = score_transaction(txn)

    approval = send_alert(
        transaction_id,
        amount,
        res["overall_risk"]
    )

    return f"""

    <html>
    <body style="font-family: Arial; padding:40px;">

        <h1>Transaction Result</h1>

        <h2>{approval["decision"]}</h2>

        <p>{approval["message"]}</p>

        <p>
        Fraud Probability:
        {round(res["fraud_probability"], 4)}
        </p>

        <p>
        Overall Risk:
        {round(res["overall_risk"], 4)}
        </p>

        <a href="/">Check Another Transaction</a>

    </body>
    </html>

    """


# ---------- API ENDPOINT ----------

@app.route("/score", methods=["POST"])
def score_api():

    payload = request.json

    if payload is None:
        return jsonify({"error": "Invalid JSON"}), 400

    txns = payload.get("transactions", [payload])

    results = {}

    for txn in txns:

        txn_id = txn.get(
            "transaction_id",
            f"txn_{int(time.time()*1000)}"
        )

        amount = txn.get("amount", 0)

        res = score_transaction(txn)

        approval = send_alert(
            txn_id,
            amount,
            res["overall_risk"]
        )

        results[txn_id] = {

            "fraud_probability":
                res["fraud_probability"],

            "iso_score":
                res["iso_score"],

            "overall_risk":
                res["overall_risk"],

            "rules_triggered":
                res["rules"]["rules_triggered"],

            "decision":
                approval["decision"],

            "message":
                approval["message"]
        }

    return jsonify({"results": results})


# ================= MAIN =================

if __name__ == "__main__":

    DATA_PATH = "creditcard.csv"

    if not (MODEL_DIR / "xgb_supervised.joblib").exists():

        train_models(DATA_PATH)

    else:

        print("Models already exist")

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False
    )

Models already exist
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.38:5000
Press CTRL+C to quit
